# 09 — Neural Inertial Odometry Training (v2 — Fixed Uncertainty Head)

**SIH PS 26168 — Intelligent Dead Reckoning**

## What changed vs v1
| Aspect | v1 (baseline) | v2 (this notebook) |
|--------|--------------|---------------------|
| Uncertainty head | Unbounded linear → exp() overflow | `tanh`-bounded logvar ∈ [-5,7] |
| Observed σ at inference | **8,508,032** (overflow) | Expected: 0.08 – 91 m |
| Checkpoint dir | `checkpoints/inertial_odometry/` | `checkpoints/nio_fixed/` |
| Results dir | `results/` | `results/nio_fixed/` |
| Original checkpoint | NEVER overwritten | ✓ preserved |

## Training Monitoring (new)
- Per-epoch: mean σ, P95 σ, NaN/Inf count
- Gradient norm tracking
- Uncertainty calibration check at end of training

## 1. Environment & GPU Preflight (HARD FAIL if no CUDA)

In [ ]:
import os, sys, json, time, datetime
from pathlib import Path
import yaml
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# ── GPU check: HARD FAIL if no CUDA ──────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError(
        '[FATAL] CUDA is not available.\n'
        'NIO training must run on Lightning AI GPU.\n'
        'Do NOT fall back to CPU — training would take hours and is not reproducible.\n'
        'Start the Lightning studio and rerun this notebook.'
    )

device = torch.device('cuda')
print(f'GPU Device   : {torch.cuda.get_device_name(0)}')
print(f'GPU Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
print(f'CUDA Version : {torch.version.cuda}')
print(f'PyTorch      : {torch.__version__}')

# Checkpoint & output directories for the FIXED model (never touch original)
CKPT_DIR    = PROJECT_ROOT / 'checkpoints' / 'nio_fixed'
RESULTS_DIR = PROJECT_ROOT / 'results'     / 'nio_fixed'
PLOTS_DIR   = PROJECT_ROOT / 'plots'       / 'nio_fixed'
for d in [CKPT_DIR, RESULTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Confirm original baseline checkpoint is still intact (never overwritten)
BASELINE_CKPT = PROJECT_ROOT / 'checkpoints' / 'inertial_odometry' / 'inertial_odometry_best.pt'
if BASELINE_CKPT.exists():
    print(f'[OK] Baseline checkpoint preserved: {BASELINE_CKPT}')
else:
    print('[WARN] Baseline checkpoint not found (may not have been trained yet)')

print(f'Fixed NIO checkpoints → {CKPT_DIR}')
print(f'Fixed NIO results     → {RESULTS_DIR}')

## 2. Load Configuration & Datasets

In [ ]:
with open(PROJECT_ROOT / 'configs' / 'training.yaml', 'r') as f:
    cfg = yaml.safe_load(f)['inertial_odometry']

# Reproducible seed
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('=== NIO v2 (Fixed Uncertainty) Training Config ===')
print(json.dumps(cfg, indent=2))

from src.datasets.inertial_odometry_dataset import InertialOdometryDataset
from src.models.inertial_odometry import NeuralInertialOdometry

# Tighter stride for more training windows (train split only)
WINDOW_SIZE = 100    # 10s @ 10 Hz
TRAIN_STRIDE = 20   # 50% overlap
VAL_STRIDE   = 30

print(f'\nLoading training dataset (window={WINDOW_SIZE}, stride={TRAIN_STRIDE})...')
train_ds = InertialOdometryDataset(split='train', window_size=WINDOW_SIZE, stride=TRAIN_STRIDE)
val_ds   = InertialOdometryDataset(split='val',   window_size=WINDOW_SIZE, stride=VAL_STRIDE)

BATCH_SIZE = cfg.get('batch_size', 64)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=2, pin_memory=True)

print(f'Train windows: {len(train_ds):,}  ({len(train_loader)} batches @ bs={BATCH_SIZE})')
print(f'Val   windows: {len(val_ds):,}  ({len(val_loader)} batches)')

## 3. Instantiate Fixed NIO Model (v2 — BoundedLogVarHead)

In [ ]:
model = NeuralInertialOdometry(
    input_dim=6,
    tcn_channels=cfg.get('tcn_channels', [64, 128, 256]),
    kernel_size=cfg.get('tcn_kernel_size', 3),
    dropout=cfg.get('dropout', 0.1),
    vel_loss_weight=cfg.get('velocity_loss_weight', 0.5),
    uncertainty_weight=cfg.get('uncertainty_loss_weight', 0.1)
).to(device)

param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable Parameters: {param_count:,}')

# Verify the logvar head is now BoundedLogVarHead
head_type = type(model.logvar_head).__name__
assert head_type == 'BoundedLogVarHead', f'Expected BoundedLogVarHead, got {head_type}'
print(f'[OK] Logvar head type: {head_type}')
print(f'[OK] Logvar bounds: [{model.logvar_head.center - model.logvar_head.scale:.1f}, {model.logvar_head.center + model.logvar_head.scale:.1f}]')
print(f'[OK] Sigma bounds: [{float(np.exp(0.5*(model.logvar_head.center - model.logvar_head.scale))):.3f}m, {float(np.exp(0.5*(model.logvar_head.center + model.logvar_head.scale))):.1f}m]')

# Sanity check: run a dummy forward pass and verify sigma is finite
dummy_x = torch.randn(4, 100, 6, device=device)
with torch.no_grad():
    d_, v_, lv_ = model(dummy_x)
    stats_ = model.compute_uncertainty_stats(lv_)
assert stats_['nan_count'] == 0, 'NaN detected in dummy forward pass!'
assert stats_['inf_count'] == 0, 'Inf detected in dummy forward pass!'
print(f'[OK] Dummy forward pass: sigma_mean={stats_["mean_sigma"]:.3f}m, sigma_max={stats_["max_sigma"]:.3f}m (no NaN/Inf)')

NUM_EPOCHS = cfg.get('num_epochs', 100)
optimizer  = torch.optim.AdamW(
    model.parameters(),
    lr=float(cfg.get('learning_rate', 1.0e-3)),
    weight_decay=float(cfg.get('weight_decay', 1.0e-4))
)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)
scaler     = torch.cuda.amp.GradScaler(enabled=True)

EARLY_STOP_PATIENCE = 15
print(f'\nTraining for up to {NUM_EPOCHS} epochs (early stop patience={EARLY_STOP_PATIENCE})')

## 4. Training Loop with Uncertainty Monitoring

In [ ]:
train_losses, val_losses = [], []
val_disp_rmses, val_vel_rmses = [], []
sigma_means, sigma_p95s = [], []
grad_norms = []
nan_counts_train, nan_counts_val = [], []

best_val_loss    = float('inf')
best_epoch       = 0
patience_counter = 0
BEST_CKPT_PATH   = CKPT_DIR / 'nio_fixed_best.pt'
LAST_CKPT_PATH   = CKPT_DIR / 'nio_fixed_last.pt'

t0 = time.time()
print(f'Training started at {datetime.datetime.utcnow().isoformat()}Z')

for epoch in range(1, NUM_EPOCHS + 1):
    # ── TRAIN ────────────────────────────────────────────────────────────────
    model.train()
    ep_loss, n_train = 0.0, 0
    ep_nan_train = 0
    ep_grad_norm = 0.0

    for batch in train_loader:
        x_imu   = batch['imu'].to(device)
        gt_disp = batch['disp'].to(device)
        gt_vel  = batch['vel'].to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=True):
            p_disp, p_vel, p_logvar = model(x_imu)
            loss, l_disp, l_vel = model.compute_loss(p_disp, p_vel, p_logvar, gt_disp, gt_vel)

        if torch.isnan(loss) or torch.isinf(loss):
            ep_nan_train += 1
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        gn = nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        ep_grad_norm += float(gn)
        scaler.step(optimizer)
        scaler.update()

        ep_loss += loss.item()
        n_train += 1

    scheduler.step()
    avg_train_loss = ep_loss / max(1, n_train)
    train_losses.append(avg_train_loss)
    grad_norms.append(ep_grad_norm / max(1, n_train))
    nan_counts_train.append(ep_nan_train)

    # ── VALIDATION ───────────────────────────────────────────────────────────
    model.eval()
    ep_val_loss, n_val = 0.0, 0
    all_disp_err, all_vel_err = [], []
    all_logvars = []
    ep_nan_val = 0

    with torch.no_grad():
        for batch in val_loader:
            x_imu   = batch['imu'].to(device)
            gt_disp = batch['disp'].to(device)
            gt_vel  = batch['vel'].to(device)

            with torch.cuda.amp.autocast(enabled=True):
                p_disp, p_vel, p_logvar = model(x_imu)
                v_loss, _, _ = model.compute_loss(p_disp, p_vel, p_logvar, gt_disp, gt_vel)

            if torch.isnan(v_loss) or torch.isinf(v_loss):
                ep_nan_val += 1
                continue

            ep_val_loss  += v_loss.item()
            n_val        += 1
            disp_err = torch.norm(p_disp - gt_disp, dim=1).cpu().numpy()
            vel_err  = torch.abs(p_vel[:, 0] - gt_vel[:, 0]).cpu().numpy()
            all_disp_err.extend(disp_err)
            all_vel_err.extend(vel_err)
            all_logvars.append(p_logvar.cpu())

    avg_val_loss = ep_val_loss / max(1, n_val)
    disp_rmse    = float(np.sqrt(np.mean(np.array(all_disp_err) ** 2))) if all_disp_err else float('nan')
    vel_rmse     = float(np.sqrt(np.mean(np.array(all_vel_err)  ** 2))) if all_vel_err  else float('nan')

    # Uncertainty stats from validation batch
    if all_logvars:
        lv_cat = torch.cat(all_logvars, dim=0)
        u_stats = model.compute_uncertainty_stats(lv_cat)
    else:
        u_stats = {'mean_sigma': float('nan'), 'p95_sigma': float('nan'),
                   'nan_count': 0, 'inf_count': 0}

    val_losses.append(avg_val_loss)
    val_disp_rmses.append(disp_rmse)
    val_vel_rmses.append(vel_rmse)
    sigma_means.append(u_stats['mean_sigma'])
    sigma_p95s.append(u_stats['p95_sigma'])
    nan_counts_val.append(ep_nan_val + u_stats.get('nan_count', 0))

    # ── CHECKPOINT ───────────────────────────────────────────────────────────
    ckpt_payload = {
        'epoch':             epoch,
        'model_state_dict':  model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_loss':     best_val_loss,
        'disp_rmse':         disp_rmse,
        'vel_rmse':          vel_rmse,
        'sigma_mean':        u_stats['mean_sigma'],
        'config':            cfg,
        'param_count':       param_count,
        'seed':              SEED,
        'uncertainty_fix':   'BoundedLogVarHead_v2',
    }

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch    = epoch
        patience_counter = 0
        ckpt_payload['best_val_loss'] = best_val_loss
        torch.save(ckpt_payload, BEST_CKPT_PATH)
        star = ' ✓ [BEST]'
    else:
        patience_counter += 1
        star = ''

    # Always save last checkpoint
    torch.save(ckpt_payload, LAST_CKPT_PATH)

    # ── LOGGING ──────────────────────────────────────────────────────────────
    if epoch % 5 == 0 or epoch == 1 or star:
        elapsed = (time.time() - t0) / 60
        print(
            f'Ep {epoch:3d}/{NUM_EPOCHS} | '
            f'Train={avg_train_loss:.4f} Val={avg_val_loss:.4f} | '
            f'Disp={disp_rmse:.2f}m Vel={vel_rmse:.2f}m/s | '
            f'σ_mean={u_stats["mean_sigma"]:.3f}m σ_P95={u_stats["p95_sigma"]:.3f}m | '
            f'NaN={ep_nan_train}tr/{ep_nan_val}val | '
            f'{elapsed:.1f}min{star}'
        )

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f'[EARLY STOP] No val improvement for {EARLY_STOP_PATIENCE} epochs. Best epoch: {best_epoch}')
        break

total_time_min = (time.time() - t0) / 60
print(f'\nTraining complete in {total_time_min:.1f} minutes')
print(f'Best checkpoint: {BEST_CKPT_PATH}  (epoch {best_epoch}, val_loss={best_val_loss:.4f})')

## 5. Training Curves + Uncertainty Monitoring Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ep_range = range(1, len(train_losses) + 1)

# Loss curves
axes[0,0].plot(ep_range, train_losses, 'b-',  label='Train Loss')
axes[0,0].plot(ep_range, val_losses,   'r--', label='Val Loss')
axes[0,0].set_title('Multi-Task Loss (NLL + Vel MSE)', fontweight='bold')
axes[0,0].set_xlabel('Epoch'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.4)

# Displacement & velocity RMSE
axes[0,1].plot(ep_range, val_disp_rmses, color='forestgreen', label='Disp RMSE (m)')
axes[0,1].plot(ep_range, val_vel_rmses,  color='darkorange',  linestyle='--', label='Vel RMSE (m/s)')
axes[0,1].axhline(63.906, color='r', linestyle=':', label='Baseline NIO RMSE=63.9m')
axes[0,1].set_title('Validation RMSE', fontweight='bold')
axes[0,1].set_xlabel('Epoch'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.4)

# Uncertainty (sigma) tracking — KEY diagnostic
axes[1,0].plot(ep_range, sigma_means, 'purple', label='σ_mean (m)')
axes[1,0].plot(ep_range, sigma_p95s,  'violet', linestyle='--', label='σ_P95 (m)')
axes[1,0].axhline(91.2, color='r', linestyle=':', label='Max physical bound=91.2m')
axes[1,0].axhline(8508032, color='gray', linestyle=':', linewidth=0.5, label='Baseline overflow=8.5M')
axes[1,0].set_title('Uncertainty σ Monitoring (FIXED)', fontweight='bold')
axes[1,0].set_xlabel('Epoch'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.4)
axes[1,0].set_yscale('log')

# Gradient norm
axes[1,1].plot(ep_range, grad_norms, 'teal', label='Avg grad norm')
axes[1,1].set_title('Gradient Norm per Epoch', fontweight='bold')
axes[1,1].set_xlabel('Epoch'); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.4)

plt.suptitle('NIO v2 — Fixed Uncertainty Head Training', fontsize=14, fontweight='bold')
plt.tight_layout()
fig_path = PLOTS_DIR / 'nio_fixed_training_curves.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.close()
print(f'Training curves saved: {fig_path}')

## 6. Save Training Summary

In [ ]:
import subprocess, platform

# Try to get git commit
try:
    git_hash = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'],
                                        cwd=str(PROJECT_ROOT)).decode().strip()
except Exception:
    git_hash = 'NOT_AVAILABLE'

# Final sigma stats from best checkpoint
best_ckpt_data = torch.load(BEST_CKPT_PATH, map_location='cpu', weights_only=False)
final_sigma_mean = best_ckpt_data.get('sigma_mean', float('nan'))

summary = {
    'model':                    'NeuralInertialOdometry_TCN_v2_FixedUncertainty',
    'experiment':               'nio_fixed_tanh_bounded_logvar',
    'timestamp':                datetime.datetime.utcnow().isoformat() + 'Z',
    'git_commit':               git_hash,
    'seed':                     SEED,
    'device':                   torch.cuda.get_device_name(0),
    'cuda_version':             torch.version.cuda,
    'pytorch_version':          torch.__version__,
    'uncertainty_fix':          'BoundedLogVarHead: logvar=6*tanh(raw)+1 → σ∈[0.082m, 91.2m]',
    'epochs_trained':           len(train_losses),
    'best_epoch':               best_epoch,
    'best_val_loss':            round(float(best_val_loss), 4),
    'final_val_disp_rmse_m':    round(float(val_disp_rmses[best_epoch-1]), 3) if val_disp_rmses else None,
    'final_val_vel_rmse_mps':   round(float(val_vel_rmses[best_epoch-1]),  3) if val_vel_rmses  else None,
    'final_sigma_mean_m':       round(float(final_sigma_mean), 4) if not np.isnan(float(final_sigma_mean)) else 'NaN',
    'total_nan_train_batches':  int(sum(nan_counts_train)),
    'total_nan_val_batches':    int(sum(nan_counts_val)),
    'trainable_params':         param_count,
    'window_size':              WINDOW_SIZE,
    'train_stride':             TRAIN_STRIDE,
    'batch_size':               BATCH_SIZE,
    'train_windows':            len(train_ds),
    'val_windows':              len(val_ds),
    'checkpoint_best':          str(BEST_CKPT_PATH.relative_to(PROJECT_ROOT)),
    'checkpoint_last':          str(LAST_CKPT_PATH.relative_to(PROJECT_ROOT)),
    'baseline_comparison': {
        'baseline_disp_rmse_m':   63.906,
        'baseline_sigma_mean':    8508032.0,
        'baseline_source':        'results/inertial_odometry_results.json'
    },
    'config': cfg
}

summary_path = RESULTS_DIR / 'train_metrics.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('=== NIO v2 Training Summary ===')
for k, v in summary.items():
    if k not in ('config', 'baseline_comparison'):
        print(f'  {k}: {v}')
print(f'\nSaved to: {summary_path}')
print('\n[CRITICAL] Verify final_sigma_mean is in [0.082, 91.2] range.')
if isinstance(summary['final_sigma_mean_m'], float):
    assert 0.0 < summary['final_sigma_mean_m'] < 200.0, f'Sigma still anomalous: {summary["final_sigma_mean_m"]}'
    print(f'[PASS] σ_mean = {summary["final_sigma_mean_m"]:.4f} m  (physically reasonable)')
else:
    print('[FAIL] σ_mean is NaN — check training logs')